# ============================================================================
# SAR RDA (Range Doppler Algorithm) 大斜视角完全体 - 包含 SRC
# ============================================================================

## 1. 算法背景与大斜视升级核心
传统的低斜视 RDA 算法基于斜距的**抛物线近似**，这在斜视角较大（通常 $> 5^\circ$）或分辨率较高时会导致严重的相位误差和距离向散焦。
为了解决这一问题，本代码实现了**大斜视角下的精确 RDA 算法**，完成了以下三大核心理论升级：
1. **二次距离压缩 (SRC)**：在二维频域消除由大斜视引起的距离-方位交叉耦合。
2. **精确的 RCMC**：废除抛物线近似，采用基于双曲线斜距方程推导的精确解析解。
3. **基于 POSP 的方位压缩**：采用驻留相位原理推导的精确解析相位匹配滤波器。

### 派生参数计算公式说明
根据基础雷达参数，可以推导出后续处理所需的关键物理量：
* **最近斜距 ($R_0$)**: 景中心斜距在零多普勒方向上的投影，公式为 $R_0 = R_{nc} \cos(\theta_{r,c})$。
* **脉冲采样点数 ($N_r$)**: 发射脉冲在离散时间上的点数，$N_r = T_r \cdot F_r$。
* **距离向带宽 ($BW_{range}$)**: 线性调频信号的总带宽，$BW_{range} = K_r \cdot T_r$。
* **雷达波长 ($\lambda$)**: $\lambda = c / f_0$。
* **多普勒中心频率 ($f_{nc}$)**: 由于斜视角 $\theta_{r,c}$ 的存在，多普勒历程的中心发生偏移，公式为 $f_{nc} = \frac{2 V_r \sin(\theta_{r,c})}{\lambda}$。
* **方位向天线长度 ($L_{a\_real}$)**: 由多普勒带宽反推的物理天线长度，$L_{a\_real} = \frac{0.886 \cdot 2 V_r \cos(\theta_{r,c})}{BW_{dop}}$。
* **雷达波束宽度 ($\beta_{bw}$)**: $\beta_{bw} = \frac{0.886 \cdot \lambda}{L_{a\_real}}$。
* **合成孔径长度 ($L_a$)**: 雷达波束扫过目标的总距离，$L_a = \frac{0.886 \cdot R_{nc} \cdot \lambda}{L_{a\_real}}$。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# ====================================================================
# 1. 定义参数
# ====================================================================
R_nc = 20e3                 # 景中心斜距
Vr = 150                    # 雷达有效速度
Tr = 2.5e-6                 # 发射脉冲时宽
Kr = 20e12                  # 距离调频率
f0 = 5.3e9                  # 雷达工作频率
BW_dop = 80                 # 多普勒带宽
Fr = 60e6                   # 距离采样率
Fa = 200                    # 方位采样率
Naz = 1024                  # 方位向采样点数
Nrg = 1024                  # 距离向采样点数
sita_r_c = (10 * np.pi) / 180  # 波束斜视角 (10度大斜视)
c = 3e8                     # 光速

R0 = R_nc * np.cos(sita_r_c)    
Nr = int(Tr * Fr)               
BW_range = Kr * Tr              
lamda = c / f0                  
fnc = 2 * Vr * np.sin(sita_r_c) / lamda  
La_real = 0.886 * 2 * Vr * np.cos(sita_r_c) / BW_dop  
beta_bw = 0.886 * lamda / La_real        
La = 0.886 * R_nc * lamda / La_real      

NFFT_r = Nrg                
NFFT_a = Naz

## 2. 大斜视几何与原始回波生成 
在斜视模式下，目标 $k$ 的波束中心穿越时刻 $\eta_c$ 为：
$$\eta_c = \frac{y_k - x_k \tan(\theta_{r,c})}{V_r}$$

目标真实的双曲线瞬时斜距方程为：
$$R_n(\eta) = \sqrt{x_k^2 + (V_r \eta - y_k)^2}$$

二维基带回波模型（包含方位双程天线方向图 $w_a$ 与距离向矩形窗 $w_r$）：
$$s(\tau,\eta) = A_0 w_r w_a \exp\left(-j\frac{4\pi f_0 R_n(\eta)}{c}\right) \exp\left(j\pi K_r\left(\tau-\frac{2R_n(\eta)}{c}\right)^2\right)$$

In [ ]:
# ====================================================================
# 2 & 3 & 4. 设定点目标与生成回波数据
# ====================================================================
delta_R0 = 0        
delta_R1 = 120      
delta_R2 = 50

x1 = R0 
y1 = delta_R0 + x1 * np.tan(sita_r_c)
x2 = x1 
y2 = y1 + delta_R1 
x3 = x2 + delta_R2 
y3 = y2 + delta_R2 * np.tan(sita_r_c) 

x_range = np.array([x1, x2, x3])
y_azimuth = np.array([y1, y2, y3])
nc_target = (y_azimuth - x_range * np.tan(sita_r_c)) / Vr

tr = 2 * x1 / c + (np.arange(-Nrg/2, Nrg/2)) / Fr     
ta = np.arange(-Naz/2, Naz/2) / Fa                    
tr_mtx, ta_mtx = np.meshgrid(tr, ta)                  

print(">> 开始生成大斜视回波数据...")
s_echo = np.zeros((Naz, Nrg), dtype=np.complex128)
A0 = 1 

for k in range(3):
    R_n = np.sqrt(x_range[k]**2 + (Vr * ta_mtx - y_azimuth[k])**2)  
    w_range = np.abs(tr_mtx - 2 * R_n / c) <= (Tr / 2)
    sita = np.arctan(Vr * (ta_mtx - nc_target[k]) / x_range[k])
    w_azimuth_sinc = (np.sinc(0.886 * sita / beta_bw))**2
    w_azimuth_rect = np.abs(ta_mtx - nc_target[k]) <= (La / 2) / Vr
    w_azimuth = w_azimuth_sinc * w_azimuth_rect
    s_k = A0 * w_range * w_azimuth * np.exp(-1j * 4 * np.pi * f0 * R_n / c) * np.exp(1j * np.pi * Kr * (tr_mtx - 2 * R_n / c)**2)
    s_echo += s_k

## 3. 教科书级联合处理：距离压缩与二次距离压缩 (SRC) 

在大斜视条件下，信号频谱发生严重倾斜耦合，距离调频率变为方位频率 $f_\eta$ 的函数。必须在**二维频域**引入 SRC 滤波器进行校正。

* **标准距离匹配滤波器 $H_{range}(f_\tau)$**：
用于压缩发射的线性调频脉冲，其相位是原始调频相位的复共轭（通常还会叠加一个 Kaiser 窗 $W_r(f_\tau)$ 来抑制旁瓣）：
$$H_{range}(f_\tau) = W_r(f_\tau) \exp\left\{j\pi \frac{f_\tau^2}{K_r}\right\}$$

* **距离徙动因子 $D(f_\eta)$** (基于驻留相位原理)：
反映了斜视造成的距离历程变化规律：
$$D(f_\eta) = \sqrt{1 - \left(\frac{\lambda f_\eta}{2V_r}\right)^2}$$

* **SRC 等效调频率倒数**：
推导出的交叉耦合补偿项，它精确解算了由于 $f_\eta$ 不同而产生的附加二次距离相位误差：
$$\frac{1}{K_{src}(f_\eta)} = \frac{R_0 \lambda^3 f_\eta^2}{2 V_r^2 c^2 D^3(f_\eta)}$$

* **二维频域联合滤波相乘**：
信号进入二维频域后，一次性完成“基础距离压缩”与“SRC交叉耦合补偿”的矩阵相乘：
$$H_{src}(f_\tau, f_\eta) = \exp\left\{-j\pi \frac{f_\tau^2}{K_{src}(f_\eta)}\right\}$$
$$S_{comp}(f_\tau, f_\eta) = S_{2D}(f_\tau, f_\eta) \cdot H_{range}(f_\tau) \cdot H_{src}(f_\tau, f_\eta)$$

### 距离压缩 (匹配滤波与补零)

* **匹配滤波**: 距离向压缩本质上是接收信号与发射脉冲副本的互相关，在频域表现为相乘：
$$S_{rc}(f_\tau, \eta) = S_{range}(f_\tau, \eta) \cdot H_{ref}^*(f_\tau)$$

* **安全补零 (Zero-padding)**: 离散傅里叶变换计算的是循环卷积。为得到正确的线性卷积并避免混叠，FFT长度 $N_{safe}$ 必须满足：
$$N_{safe} \ge N_{signal} + N_{filter} - 1$$

* **加窗 (Windowing)**: 为了抑制距离压缩后高昂的 $\text{Sinc}$ 函数旁瓣，对频域参考函数施加 Kaiser 窗 $w_{kaiser}$：
$$H_{ref}^*(f_\tau) = \text{FFT}\{ s_{ref}(\tau) \cdot w_{kaiser} \}^*$$

* **卷积完成 (IFFT)**: 
$$s_{rc}(\tau, \eta) = \text{IFFT}\{ S_{rc}(f_\tau, \eta) \}$$

In [ ]:
# ====================================================================
# 5 & 6.距离压缩(驻留频域) -> 方位FFT -> SRC -> 距离IFFT
# ====================================================================
print(">> 执行基带搬移与距离向 FFT...")
N_safe = 2048

# 步骤 A: 方位向基带搬移 (时域)
s_echo_base = s_echo * np.exp(-1j * 2 * np.pi * fnc * np.tile(ta[:, np.newaxis], (1, Nrg)))

# 步骤 B: 距离向 FFT (进入距离频域-方位时域)
S_range_freq = np.fft.fft(s_echo_base, n=N_safe, axis=1)

# 步骤 C: 距离压缩 (只乘滤波器，不进行 IFFT，保留在距离频域)
print(">> 执行距离压缩 (驻留频域)...")
fr = np.fft.fftfreq(N_safe, d=1/Fr)
t_ref = np.arange(-Nr/2, Nr/2) / Fr
s_ref = np.exp(1j * np.pi * Kr * (t_ref**2)) * np.kaiser(Nr, 2.5)
S_ref = np.fft.fft(s_ref, n=N_safe)
H_range = np.conj(S_ref)[np.newaxis, :] 

S_rc_freq = S_range_freq * H_range 

# 步骤 D: 方位向 FFT (此时正式进入二维频域)
print(">> 执行方位向 FFT，进入二维频域...")
S_2D = np.fft.fft(S_rc_freq, n=NFFT_a, axis=0)

# 步骤 E: 二次距离压缩 (SRC)
print(">> 引入 SRC 滤波器进行交叉耦合修正...")
fa = fnc + np.fft.fftfreq(NFFT_a, d=1/Fa)
fr_mtx, fa_mtx = np.meshgrid(fr, fa)

D_fa = np.sqrt(1 - (lamda * fa_mtx / (2 * Vr))**2)
inv_K_src = (R0 * lamda**3 * fa_mtx**2) / (2 * Vr**2 * c**2 * D_fa**3)
H_src = np.exp(-1j * np.pi * (fr_mtx**2) * inv_K_src)

S_2D_comp = S_2D * H_src 

# 步骤 F: 距离向 IFFT (退回到距离多普勒域，准备 RCMC)
print(">> 执行距离向 IFFT，退回距离多普勒域...")
s_rc = np.fft.ifft(S_2D_comp, axis=1)
N_rg = Nrg - Nr + 1
S_rd = s_rc[:, :N_rg]

## 4. 解析化 RCMC 与方位聚焦 
* **精确的距离徙动方程 (RCMC)**：
废弃 $\frac{\lambda^2R_0f_\eta^2}{8V_r^2}$ 抛物线近似，采用无近似解析解：
$$\Delta R(f_\eta) = R_0 \left( \frac{1}{D(f_\eta)} - 1 \right)$$
算法使用 $8$ 点 Sinc 插值核完成亚像素级轨迹矫正。

* **精确的方位匹配滤波器**：
同样废弃基于调频率 $K_a$ 的二次型近似，采用基于驻留相位原理的严格复共轭相位：
$$H_{az}(f_\eta) = \exp\left\{j \frac{4\pi R_0}{\lambda} D(f_\eta)\right\}$$

In [ ]:
# ====================================================================
# 7. 距离徙动校正 (RCMC)
# ====================================================================
print(">> 开始精确 RCMC (基于双曲线解析方程)...")

D_fa_1D = np.sqrt(1 - (lamda * fa / (2 * Vr))**2)
tr_RCMC = 2 * x1 / c + np.arange(-N_rg/2, N_rg/2) / Fr
R0_RCMC = (c / 2) * tr_RCMC * np.cos(sita_r_c)

# 采用非近似的双曲线 RCM 解析公式
delta_Rrd_fn = R0_RCMC * (1 / D_fa_1D[:, np.newaxis] - 1)

num_range = c / (2 * Fr)
delta_Rrd_fn_num = delta_Rrd_fn / num_range

R = 8
S_rd_rcmc = np.zeros((NFFT_a, N_rg), dtype=np.complex128)
pos = np.arange(N_rg)[np.newaxis, :] + delta_Rrd_fn_num
pos_ceil = np.ceil(pos).astype(int)

weights = np.zeros((R, NFFT_a, N_rg))
pts_wrap = np.zeros((R, NFFT_a, N_rg), dtype=int)

for idx, k in enumerate(range(-R//2 + 1, R//2 + 1)): 
    pts = pos_ceil - k
    weights[idx] = np.sinc(pos - pts) 
    pts_wrap[idx] = pts % N_rg        

weights /= np.sum(weights, axis=0)
for idx in range(R):
    S_rd_rcmc += weights[idx] * S_rd[np.arange(NFFT_a)[:, np.newaxis], pts_wrap[idx]]

# ====================================================================
# 8. 方位压缩
# ====================================================================
print(">> 开始基于解析相位的方位向压缩...")

Haz = np.exp(1j * 4 * np.pi * R0_RCMC / lamda * D_fa_1D[:, np.newaxis])
S_rd_c = S_rd_rcmc * Haz
s_ac = np.fft.ifft(S_rd_c, axis=0)

# ====================================================================
# 9. 统一绘图函数
# ====================================================================
print(">> 渲染图像中...")
def plot_img(data, title, idx):
    plt.figure(idx, figsize=(6, 5))
    plt.imshow(np.abs(data), aspect='auto', cmap='gray')
    plt.title(title)
    plt.xlabel('距离向 (采样点)')
    plt.ylabel('方位向 (采样点)')
    plt.colorbar()
    plt.tight_layout()

plot_img(s_echo, '图1：原始雷达回波数据', 1)
plot_img(S_rd,   '图2：距离多普勒域 (已做SRC，未 RCMC)', 2)
plot_img(S_rd_rcmc, '图3：距离多普勒域 (已进行矢量化 Sinc RCMC)', 3)
plot_img(s_ac,   '图4：最终 SAR 聚焦成像', 4)
plt.show()

## 5. 图形显示与量化指标评估
高精度成像算法必须通过严格的点目标质量分析：
1. **冲激响应宽度 (IRW)**: 主瓣下降到一半能量时的物理宽度（$-3\text{ dB}$）。
2. **峰值旁瓣比 (PSLR)**: 最高旁瓣能量相对于主瓣最高能量的对数比（理想 Sinc 约为 $-13.26\text{ dB}$，加 Kaiser 窗后压制至 $-20\text{ dB}$ 左右）。
3. **积分旁瓣比 (ISLR)**: 所有旁瓣的总能量与主瓣总能量的比值。

In [ ]:
# ====================================================================
# 10. 大斜视点目标质量分析
# ====================================================================
print(">> 开始对 3 个点目标进行量化指标评估...")
from scipy.signal import find_peaks

def analyze_1d_target(slice_1d, pixel_spacing, up_factor=16):
    N = len(slice_1d)
    F = np.fft.fftshift(np.fft.fft(np.fft.fftshift(slice_1d)))
    pad_len = N * up_factor
    F_pad = np.pad(F, (pad_len//2 - N//2, pad_len//2 - (N - N//2)), 'constant')
    slice_up = np.abs(np.fft.fftshift(np.fft.ifft(np.fft.fftshift(F_pad)))) * up_factor
    
    res_up = pixel_spacing / up_factor
    
    peak_idx = np.argmax(slice_up)
    peak_val = slice_up[peak_idx]
    slice_norm = slice_up / peak_val
    slice_db = 20 * np.log10(slice_norm + 1e-12)
    
    crossings = np.where(np.diff(np.sign(slice_db + 3)))[0]
    left_crosses = crossings[crossings < peak_idx]
    right_crosses = crossings[crossings >= peak_idx]
    
    if len(left_crosses) == 0 or len(right_crosses) == 0:
        return 0.0, 0.0, 0.0, slice_db, res_up, peak_idx
        
    left_idx, right_idx = left_crosses[-1], right_crosses[0]
    
    x_left = left_idx + (-3 - slice_db[left_idx]) / (slice_db[left_idx+1] - slice_db[left_idx])
    x_right = right_idx + (-3 - slice_db[right_idx]) / (slice_db[right_idx+1] - slice_db[right_idx])
    irw = (x_right - x_left) * res_up
    
    null_width = int((x_right - x_left) * 1.5)
    peaks, _ = find_peaks(slice_norm)
    sidelobes = [p for p in peaks if p < peak_idx - null_width or p > peak_idx + null_width]
    pslr = np.max(slice_db[sidelobes]) if len(sidelobes) > 0 else 0
    
    null_left = max(0, peak_idx - null_width)
    null_right = min(len(slice_norm)-1, peak_idx + null_width)
    E_main = np.sum(slice_norm[null_left:null_right]**2)
    E_total = np.sum(slice_norm**2)
    E_side = E_total - E_main
    islr = 10 * np.log10(E_side / E_main) if E_side > 0 else 0
    
    return irw, pslr, islr, slice_db, res_up, peak_idx

print("\n" + "="*55)
print("大斜视点目标成像质量评估报告 (3点连测)")
print("="*55)

dx = c / (2 * Fr) 
dy = Vr / Fa       

SAR_abs = np.abs(s_ac)
temp_img = SAR_abs.copy()
max_y, max_x = SAR_abs.shape  

WIN = 16 

for target_id in range(3):
    center_y, center_x = np.unravel_index(np.argmax(temp_img), temp_img.shape)
    
    slice_azimuth = SAR_abs[max(0, center_y-WIN):min(max_y, center_y+WIN), center_x]
    slice_range = SAR_abs[center_y, max(0, center_x-WIN):min(max_x, center_x+WIN)]
    
    irw_r, pslr_r, islr_r, db_r, res_r, p_r = analyze_1d_target(slice_range, dx)
    irw_a, pslr_a, islr_a, db_a, res_a, p_a = analyze_1d_target(slice_azimuth, dy)
    
    print(f"▶ 目标 {target_id + 1} (位于矩阵坐标 Y:{center_y}, X:{center_x})")
    print(f"  [距离向 Range]   IRW: {irw_r:.3f} m | PSLR: {pslr_r:.2f} dB | ISLR: {islr_r:.2f} dB")
    print(f"  [方位向 Azimuth] IRW: {irw_a:.3f} m | PSLR: {pslr_a:.2f} dB | ISLR: {islr_a:.2f} dB")
    print("-" * 55)
    
    mask = 8
    temp_img[max(0, center_y-mask):min(max_y, center_y+mask), max(0, center_x-mask):min(max_x, center_x+mask)] = 0

print("✅ 所有目标测试完毕！")
print("="*55 + "\n")